# Spendly frozen V2 — final untouched-seed test

This is the one-time generalization gate before any app integration. It loads the exact frozen
forecast, anomaly model, scaler, feature schema, rules, weights, and thresholds exported by
`Spending_Model_Development_V2`.

The final 2026 dataset uses new user IDs and a seed derived deterministically from the submitted
V2 evidence **before scoring**. There is no candidate search, threshold tuning, or model fitting
in this notebook. Final labels are consulted only after predictions have been frozen.


## How to run

1. Open this notebook in Google Colab and choose **Runtime → Run all**.
2. Upload these three files if prompted:
   - `combined_finance_raw.csv`
   - `spendly_generalization_output.zip`
   - `spendly_development_v2_output.zip`
3. Download `spendly_final_seed_output.zip`.
4. Paste the block between `SPENDLY_FINAL_SEED_RESULT_START` and
   `SPENDLY_FINAL_SEED_RESULT_END` back into the task.

Expected CPU runtime is approximately 5–9 minutes. Do not change the seed, configuration, or
acceptance targets after seeing results.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "joblib": "joblib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependency check: PASS")


In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import math
import os
import shutil
import sys
import time
import warnings
import zipfile
from collections import defaultdict, deque
from datetime import datetime, time as clock_time, timedelta
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.metrics import (
    confusion_matrix, mean_squared_error, precision_recall_curve,
    precision_recall_fscore_support, r2_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
LOOKBACK = 8
RECURRING_CATEGORIES = ["Rent", "Utilities", "Subscriptions", "Savings"]
KNOWN_RECURRING = set(RECURRING_CATEGORIES)
FINAL_START = pd.Timestamp("2026-01-01")
FINAL_DAYS = 365
FINAL_USER_COUNT = 500
OUTPUT_DIR = Path("spendly_final_seed_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
ACCEPTANCE = {
    "forecast_wape_max": 30.0,
    "forecast_r2_min": 0.50,
    "anomaly_precision_min": 0.60,
    "anomaly_recall_min": 0.75,
}
print({"lookback_weeks": LOOKBACK, "final_users": FINAL_USER_COUNT, "targets": ACCEPTANCE})


In [ ]:
EXPECTED_ORIGINAL_SHA256 = "b39cb5e5725a504c8880358edd160945323e208625af441a8680decd0f34b834"
EXPECTED_DEVELOPMENT_STRESS_SHA256 = "aca051084a166f5dc81060fb189f50deb2caddd01a93ebca4759306606755b75"

def locate(env_name: str, filename: str) -> Path:
    requested = Path(os.environ.get(env_name, filename))
    if requested.exists():
        return requested
    matches = list(Path.cwd().rglob(filename))
    return matches[0] if matches else requested


original_path = locate("SPENDLY_DATA_PATH", "combined_finance_raw.csv")
generalization_zip_path = locate("SPENDLY_GENERALIZATION_ZIP", "spendly_generalization_output.zip")
v2_zip_path = locate("SPENDLY_V2_ZIP", "spendly_development_v2_output.zip")

if not all(path.exists() for path in [original_path, generalization_zip_path, v2_zip_path]):
    try:
        from google.colab import files
        files.upload()
        original_path = locate("SPENDLY_DATA_PATH", "combined_finance_raw.csv")
        generalization_zip_path = locate("SPENDLY_GENERALIZATION_ZIP", "spendly_generalization_output.zip")
        v2_zip_path = locate("SPENDLY_V2_ZIP", "spendly_development_v2_output.zip")
    except ImportError:
        pass

missing_inputs = [str(path) for path in [original_path, generalization_zip_path, v2_zip_path] if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Missing required input files: {missing_inputs}")

original_sha256 = hashlib.sha256(original_path.read_bytes()).hexdigest()
assert original_sha256 == EXPECTED_ORIGINAL_SHA256, "Original dataset hash does not match the locked input"
original = pd.read_csv(original_path, low_memory=False)
original["transaction_timestamp"] = pd.to_datetime(original["transaction_timestamp"])

def member_by_suffix(archive: zipfile.ZipFile, suffix: str) -> str:
    matches = [name for name in archive.namelist() if name.endswith(suffix)]
    if len(matches) != 1:
        raise ValueError(f"Expected one archive member ending in {suffix!r}, found {matches}")
    return matches[0]


with zipfile.ZipFile(generalization_zip_path) as archive:
    assert archive.testzip() is None, "Development-stress archive failed CRC validation"
    stress_member = member_by_suffix(archive, "independent_stress_finance.csv")
    development_stress_bytes = archive.read(stress_member)
development_stress_sha256 = hashlib.sha256(development_stress_bytes).hexdigest()
assert development_stress_sha256 == EXPECTED_DEVELOPMENT_STRESS_SHA256, "Development stress dataset hash mismatch"
development_stress = pd.read_csv(io.BytesIO(development_stress_bytes), low_memory=False)
development_stress["transaction_timestamp"] = pd.to_datetime(development_stress["transaction_timestamp"])

with zipfile.ZipFile(v2_zip_path) as archive:
    assert archive.testzip() is None, "V2 evidence archive failed CRC validation"
    results_bytes = archive.read(member_by_suffix(archive, "spendly_development_v2_results.json"))
    schema_bytes = archive.read(member_by_suffix(archive, "v2_feature_schema.json"))
    forecast_model_bytes = archive.read(member_by_suffix(archive, "frozen_v2_forecast.joblib"))
    scaler_bytes = archive.read(member_by_suffix(archive, "frozen_v2_anomaly_scaler.joblib"))
    isolation_forest_bytes = archive.read(member_by_suffix(archive, "frozen_v2_isolation_forest.joblib"))

v2_results = json.loads(results_bytes)
feature_schema = json.loads(schema_bytes)
try:
    # scikit-learn 1.6.1 may serialize its compiled loss extension under this short module name.
    import sklearn._loss._loss as sklearn_loss_extension
    sys.modules.setdefault("_loss", sklearn_loss_extension)
except ImportError:
    pass
frozen_forecast_model = joblib.load(io.BytesIO(forecast_model_bytes))
frozen_anomaly_scaler = joblib.load(io.BytesIO(scaler_bytes))
frozen_isolation_forest = joblib.load(io.BytesIO(isolation_forest_bytes))

assert v2_results["ready_for_final_seed_test"] is True
assert v2_results["checks"]["app_changed"] is False
assert v2_results["frozen_forecast_model"] == "hist_absolute_ratio_leaf31"
assert v2_results["frozen_anomaly_if"] == "if_4096"
assert feature_schema["anomaly_hybrid_threshold"] == v2_results["anomaly_validation_metrics"]["threshold"]

v2_evidence_sha256 = hashlib.sha256(results_bytes).hexdigest()
seed_material = f"{original_sha256}|{v2_evidence_sha256}|SPENDLY_FINAL_V2_ONCE"
FINAL_SEED = int(hashlib.sha256(seed_material.encode("utf-8")).hexdigest()[:8], 16)
artifact_hashes = {
    "forecast_model_sha256": hashlib.sha256(forecast_model_bytes).hexdigest(),
    "anomaly_scaler_sha256": hashlib.sha256(scaler_bytes).hexdigest(),
    "isolation_forest_sha256": hashlib.sha256(isolation_forest_bytes).hexdigest(),
    "feature_schema_sha256": hashlib.sha256(schema_bytes).hexdigest(),
}
print({
    "input_validation": "PASS",
    "v2_evidence_sha256": v2_evidence_sha256,
    "locked_final_seed": FINAL_SEED,
    "frozen_forecast": v2_results["frozen_forecast_model"],
    "frozen_anomaly": v2_results["frozen_anomaly_if"],
})


## 1. Generate the locked final dataset

The seed is a deterministic hash of the original-data hash, the submitted V2 evidence hash, and
a fixed protocol tag. This prevents choosing a favorable seed after observing performance. The
generator retains the seen and unseen anomaly families from the independent test, but uses 500
new `FINAL-U...` users over the complete 2026 calendar year.


In [ ]:
STRESS_CATEGORIES = [
    "Airtime and Data", "Education", "Entertainment", "Food", "Healthcare",
    "Irregular Expenses", "Rent", "Savings", "Subscriptions", "Transport", "Utilities",
]
DISCRETIONARY_CATEGORIES = [
    "Airtime and Data", "Education", "Entertainment", "Food", "Healthcare",
    "Irregular Expenses", "Transport",
]
MERCHANTS = {
    "Airtime and Data": ["Safaricom", "Airtel", "Telkom"],
    "Education": ["Bookshop", "Course provider", "Campus store"],
    "Entertainment": ["Cinema", "Streaming", "Social outing"],
    "Food": ["Cafe", "Restaurant", "Supermarket", "Kibanda"],
    "Healthcare": ["Clinic", "Pharmacy", "Hospital"],
    "Irregular Expenses": ["Repair shop", "ATM", "Electronics store"],
    "Rent": ["Landlord transfer"], "Savings": ["Savings transfer"],
    "Subscriptions": ["Streaming", "Cloud service", "Gym"],
    "Transport": ["Matatu", "Fuel station", "Ride hailing"],
    "Utilities": ["KPLC", "Water company", "Gas supplier"],
}
SEEN_ANOMALIES = {
    "random_financial_shock", "unusually_large_category_expenditure",
    "sudden_transaction_frequency_increase", "duplicate_like_spending",
    "unexpected_recurring_expense_increase", "unusually_high_weekend_spending",
    "sudden_category_change",
}
UNSEEN_ANOMALIES = {
    "nighttime_cash_out", "merchant_hopping_burst", "split_transaction_evasion",
}

def income_band(income):
    if income < 30000:
        return "low"
    if income < 70000:
        return "middle"
    return "high"

def stress_base_amount(category, income, rng):
    scales = {
        "Airtime and Data": (0.002, 0.006), "Education": (0.004, 0.025),
        "Entertainment": (0.003, 0.016), "Food": (0.002, 0.012),
        "Healthcare": (0.003, 0.025), "Irregular Expenses": (0.010, 0.060),
        "Transport": (0.002, 0.010),
    }
    low, high = scales[category]
    return float(income * rng.lognormal(np.log((low + high) / 2), 0.45))

def generate_final_seed_dataset(user_count=400, start=FINAL_START, days=365, seed=FINAL_SEED):
    rng = np.random.default_rng(seed)
    records = []
    transaction_counter = 0

    def add_transaction(
        profile, timestamp, amount, category, transaction_type, merchant,
        payment_method, is_recurring=False, anomaly_type=None,
    ):
        nonlocal transaction_counter
        transaction_counter += 1
        records.append({
            "transaction_id": f"FINAL-T{transaction_counter:09d}",
            "user_id": profile["user_id"],
            "transaction_timestamp": pd.Timestamp(timestamp),
            "amount": round(max(1.0, float(amount)), 2),
            "currency": "KES", "category": category,
            "transaction_type": transaction_type, "merchant": merchant,
            "payment_method": payment_method, "is_recurring": bool(is_recurring),
            "source_dataset": "locked_final_seed_generator_v2", "is_synthetic": True,
            "user_age": profile["age"], "user_monthly_income": profile["base_income"],
            "user_employment_status": profile["status"],
            "user_income_day": profile["income_day"],
            "is_anomaly": anomaly_type is not None,
            "anomaly_type": anomaly_type,
            "label_source": "controlled_final_seed_label" if anomaly_type else None,
        })

    start_date = pd.Timestamp(start).date()
    end_date = start_date + timedelta(days=days - 1)
    seen = sorted(SEEN_ANOMALIES - {"random_financial_shock"})
    unseen = sorted(UNSEEN_ANOMALIES)

    for user_index in range(user_count):
        status = str(rng.choice(["student", "employed", "self_employed"], p=[0.36, 0.46, 0.18]))
        if status == "student":
            base_income = float(np.clip(rng.lognormal(np.log(26000), 0.38), 12000, 70000))
        elif status == "self_employed":
            base_income = float(np.clip(rng.lognormal(np.log(65000), 0.55), 18000, 220000))
        else:
            base_income = float(np.clip(rng.lognormal(np.log(52000), 0.40), 20000, 180000))
        profile = {
            "user_id": f"FINAL-U{user_index + 1:05d}", "age": int(rng.integers(18, 36)),
            "status": status, "base_income": base_income,
            "income_day": int(rng.choice([5, 15, 25, 28])),
            "rent_day": int(rng.integers(1, 6)), "utilities_day": int(rng.integers(8, 14)),
            "subscriptions_day": int(rng.integers(14, 19)), "savings_day": int(rng.integers(23, 29)),
        }
        rent = base_income * (0.16 if status == "student" else 0.23) * rng.uniform(0.80, 1.20)
        utilities = base_income * rng.uniform(0.03, 0.075)
        subscriptions = base_income * rng.uniform(0.006, 0.028)
        savings = base_income * rng.uniform(0.05, 0.20)
        base_daily_rate = 0.68 if status == "student" else 0.96
        annual_trend = float(rng.uniform(-0.04, 0.18))
        change_day = start_date + timedelta(days=int(rng.integers(120, 280)))
        income_change = float(rng.uniform(0.72, 1.30)) if rng.random() < 0.25 else 1.0
        schedule_shift = int(rng.choice([-2, -1, 1, 2])) if rng.random() < 0.20 else 0

        scenario_type = None
        scenario_day = None
        if rng.random() < 0.58:
            scenario_type = str(rng.choice(seen + unseen))
            scenario_day = start_date + timedelta(days=int(rng.integers(45, days - 10)))
            if scenario_type == "unusually_high_weekend_spending":
                while scenario_day.weekday() < 5:
                    scenario_day += timedelta(days=1)

        current = start_date
        while current <= end_date:
            progress = (current - start_date).days / max(1, days - 1)
            current_income = base_income * (income_change if current >= change_day else 1.0)
            inflation = 1.0 + annual_trend * progress
            weekend = current.weekday() >= 5
            shifted = schedule_shift if current >= change_day else 0

            if current.day == min(profile["income_day"], 28) and rng.random() > 0.06:
                add_transaction(
                    profile, datetime.combine(current, clock_time(8, 0)),
                    current_income * rng.normal(1.0, 0.08 if status != "self_employed" else 0.22),
                    "Income", "income", "Employer" if status != "self_employed" else "Client payment",
                    "Bank Transfer", True,
                )
            recurring_schedule = [
                (max(1, min(28, profile["rent_day"] + shifted)), rent, "Rent", "Landlord transfer", 0.97),
                (max(1, min(28, profile["utilities_day"] + shifted)), utilities * rng.normal(1, 0.12), "Utilities", str(rng.choice(MERCHANTS["Utilities"])), 0.93),
                (max(1, min(28, profile["subscriptions_day"] + shifted)), subscriptions, "Subscriptions", str(rng.choice(MERCHANTS["Subscriptions"])), 0.96),
                (max(1, min(28, profile["savings_day"] + shifted)), savings, "Savings", "Savings transfer", 0.82),
            ]
            for scheduled_day, recurring_amount, category, merchant, payment_probability in recurring_schedule:
                if current.day == scheduled_day and rng.random() < payment_probability:
                    amount = recurring_amount * inflation
                    anomaly = None
                    if scenario_type == "unexpected_recurring_expense_increase" and scenario_day:
                        if current.year == scenario_day.year and current.month == scenario_day.month and category == "Rent":
                            amount *= 1.65
                            anomaly = scenario_type
                    add_transaction(
                        profile, datetime.combine(current, clock_time(int(rng.integers(7, 20)), 0)),
                        amount, category, "expense", merchant,
                        "Bank Transfer" if category == "Savings" else "M-Pesa", True, anomaly,
                    )

            seasonal = 1.15 if current.month == 12 else 1.0
            daily_rate = base_daily_rate * (1.30 if weekend else 1.0)
            daily_count = int(rng.poisson(daily_rate))
            probabilities = np.array([0.08, 0.08 if status == "student" else 0.03, 0.12, 0.32, 0.07, 0.04, 0.29])
            probabilities = probabilities / probabilities.sum()
            for _ in range(daily_count):
                category = str(rng.choice(DISCRETIONARY_CATEGORIES, p=probabilities))
                amount = stress_base_amount(category, current_income, rng) * inflation * seasonal
                if weekend and category == "Entertainment":
                    amount *= 1.45
                if current.month in {1, 8} and category == "Education":
                    amount *= 1.65
                hour = int(rng.integers(6, 23))
                minute = int(rng.integers(0, 60))
                add_transaction(
                    profile, datetime.combine(current, clock_time(hour, minute)), amount,
                    category, "expense", str(rng.choice(MERCHANTS[category])),
                    str(rng.choice(["M-Pesa", "Debit Card", "Cash", "Bank Transfer"], p=[0.48, 0.24, 0.22, 0.06])),
                )

            if rng.random() < 1 / 260:
                add_transaction(
                    profile, datetime.combine(current, clock_time(16, int(rng.integers(0, 60)))),
                    current_income * rng.uniform(0.16, 0.48), "Irregular Expenses", "expense",
                    str(rng.choice(MERCHANTS["Irregular Expenses"])), "M-Pesa", False,
                    "random_financial_shock",
                )

            if scenario_day == current and scenario_type and scenario_type != "unexpected_recurring_expense_increase":
                if scenario_type == "unusually_large_category_expenditure":
                    category = str(rng.choice(["Food", "Entertainment", "Education"]))
                    add_transaction(profile, datetime.combine(current, clock_time(14, 20)), current_income * rng.uniform(0.42, 0.82), category, "expense", str(rng.choice(MERCHANTS[category])), "M-Pesa", False, scenario_type)
                elif scenario_type == "sudden_transaction_frequency_increase":
                    for event in range(7):
                        add_transaction(profile, datetime.combine(current, clock_time(12, event * 7)), rng.uniform(80, 650), "Food", "expense", "Cafe", "M-Pesa", False, scenario_type)
                elif scenario_type == "duplicate_like_spending":
                    duplicate_amount = round(rng.uniform(500, 2800), 2)
                    for minute_offset in [0, 2]:
                        add_transaction(profile, datetime.combine(current, clock_time(14, 10 + minute_offset)), duplicate_amount, "Food", "expense", "Restaurant", "M-Pesa", False, scenario_type)
                elif scenario_type == "unusually_high_weekend_spending":
                    for event in range(3):
                        add_transaction(profile, datetime.combine(current, clock_time(17 + event, 10)), current_income * rng.uniform(0.07, 0.15), "Entertainment", "expense", "Social outing", "M-Pesa", False, scenario_type)
                elif scenario_type == "sudden_category_change":
                    category = "Healthcare" if status == "student" else "Education"
                    add_transaction(profile, datetime.combine(current, clock_time(13, 15)), current_income * rng.uniform(0.16, 0.34), category, "expense", str(rng.choice(MERCHANTS[category])), "M-Pesa", False, scenario_type)
                elif scenario_type == "nighttime_cash_out":
                    for event in range(3):
                        add_transaction(profile, datetime.combine(current, clock_time(2, 5 + event * 8)), current_income * rng.uniform(0.06, 0.14), "Irregular Expenses", "expense", "ATM", "Cash", False, scenario_type)
                elif scenario_type == "merchant_hopping_burst":
                    merchants = ["Cafe", "Restaurant", "Supermarket", "Kibanda", "Food delivery"]
                    for event, merchant in enumerate(merchants):
                        add_transaction(profile, datetime.combine(current, clock_time(19, event * 9)), rng.uniform(200, 1200), "Food", "expense", merchant, "M-Pesa", False, scenario_type)
                elif scenario_type == "split_transaction_evasion":
                    for event in range(4):
                        add_transaction(profile, datetime.combine(current, clock_time(11, 5 + event * 11)), current_income * rng.uniform(0.055, 0.085), "Irregular Expenses", "expense", "Electronics store", "Debit Card", False, scenario_type)
            current += timedelta(days=1)

    stress = pd.DataFrame(records).sort_values(
        ["user_id", "transaction_timestamp", "transaction_id"]
    ).reset_index(drop=True)
    return stress


In [ ]:
generation_started = time.perf_counter()
final_seed = generate_final_seed_dataset(
    user_count=FINAL_USER_COUNT, start=FINAL_START, days=FINAL_DAYS, seed=FINAL_SEED
)
final_seed["transaction_timestamp"] = pd.to_datetime(final_seed["transaction_timestamp"])
final_seed_path = OUTPUT_DIR / "locked_final_seed_finance.csv"
final_seed.to_csv(final_seed_path, index=False)
final_seed_sha256 = hashlib.sha256(final_seed_path.read_bytes()).hexdigest()

final_user_ids = set(final_seed["user_id"].astype(str))
original_user_ids = set(original["user_id"].astype(str))
development_user_ids = set(development_stress["user_id"].astype(str))
dataset_checks = {
    "new_users_vs_original": final_user_ids.isdisjoint(original_user_ids),
    "new_users_vs_development": final_user_ids.isdisjoint(development_user_ids),
    "full_year_coverage": bool(
        final_seed["transaction_timestamp"].max() - final_seed["transaction_timestamp"].min()
        >= pd.Timedelta(days=360)
    ),
    "unique_transaction_ids": bool(final_seed["transaction_id"].is_unique),
    "positive_amounts": bool(final_seed["amount"].gt(0).all()),
    "seen_anomalies_present": bool(final_seed["anomaly_type"].isin(SEEN_ANOMALIES).any()),
    "unseen_anomalies_present": bool(final_seed["anomaly_type"].isin(UNSEEN_ANOMALIES).any()),
}
assert all(dataset_checks.values()), dataset_checks
final_seed_summary = {
    "sha256": final_seed_sha256,
    "seed": FINAL_SEED,
    "rows": int(len(final_seed)),
    "users": int(final_seed["user_id"].nunique()),
    "start": str(final_seed["transaction_timestamp"].min()),
    "end": str(final_seed["transaction_timestamp"].max()),
    "anomaly_rows": int(final_seed["is_anomaly"].sum()),
    "seen_anomaly_rows": int(final_seed["anomaly_type"].isin(SEEN_ANOMALIES).sum()),
    "unseen_anomaly_rows": int(final_seed["anomaly_type"].isin(UNSEEN_ANOMALIES).sum()),
    "generation_seconds": time.perf_counter() - generation_started,
}
display(pd.Series(final_seed_summary, name="value").to_frame())
display(final_seed.loc[final_seed["is_anomaly"], "anomaly_type"].value_counts().rename("rows").to_frame())
print("Locked final-seed quality checks: PASS")


## 2. Frozen forecast evaluation

The saved `hist_absolute_ratio_leaf31` artifact predicts directly. Features use full
Monday–Sunday weeks, eight past weeks, income normalization, and past-only recurring-schedule
inference. No `.fit()` or candidate selection occurs.


In [ ]:
def bools(series: pd.Series) -> pd.Series:
    return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})


def full_week_bounds(data: pd.DataFrame) -> tuple[pd.Timestamp, pd.Timestamp]:
    min_day = data["transaction_timestamp"].min().normalize()
    max_day = data["transaction_timestamp"].max().normalize()
    first_period = min_day.to_period("W-SUN").start_time
    last_period = max_day.to_period("W-SUN").start_time
    first_full = first_period if first_period >= min_day else first_period + pd.Timedelta(days=7)
    last_full = last_period if last_period + pd.Timedelta(days=6) <= max_day else last_period - pd.Timedelta(days=7)
    return first_full, last_full


def next_due_features(times: pd.DatetimeIndex, amounts: np.ndarray, target_start: pd.Timestamp, target_end: pd.Timestamp, income: float) -> list[float]:
    if len(times) == 0:
        return [0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
    times = pd.DatetimeIndex(times[-8:])
    amounts = np.asarray(amounts[-8:], dtype=float)
    n = len(times)
    expected_amount = float(np.median(amounts[-3:]))
    if n >= 2:
        intervals = np.diff(times.asi8) / 86_400_000_000_000.0
        recent_intervals = intervals[-4:]
        regularity = float(np.exp(-np.std(recent_intervals) / 12.0))
        next_due = times[-1] + pd.DateOffset(months=1)
    else:
        regularity = 0.35
        next_due = times[-1] + pd.DateOffset(months=1)
    while next_due < target_start:
        next_due = next_due + pd.DateOffset(months=1)
    due_flag = float((next_due >= target_start) and (next_due <= target_end))
    month_span = max(1, (times[-1].year - times[0].year) * 12 + times[-1].month - times[0].month + 1)
    payment_rate = min(1.0, n / month_span)
    confidence = min(1.0, n / 3.0) * regularity * payment_rate
    amount_ratio = expected_amount / max(income, 1.0)
    days_since = min(90.0, max(0.0, (target_start - times[-1]).total_seconds() / 86400.0)) / 90.0
    next_day_angle = 2 * math.pi * (next_due.day - 1) / 31
    return [
        due_flag, amount_ratio, confidence, days_since,
        math.sin(next_day_angle), math.cos(next_day_angle),
    ]


def build_samples(data: pd.DataFrame, source: str) -> dict[str, np.ndarray]:
    expense = data[data["transaction_type"].eq("expense")].copy()
    expense["period_start"] = expense["transaction_timestamp"].dt.to_period("W-SUN").dt.start_time
    expense["nonrec_amount"] = np.where(~expense["is_recurring"], expense["amount"], 0.0)
    expense["rec_amount"] = np.where(expense["is_recurring"], expense["amount"], 0.0)
    first_full, last_full = full_week_bounds(data)
    categories = sorted(set(data["category"]) - {"Income"})
    expected_categories = [
        "Airtime and Data", "Education", "Entertainment", "Food", "Healthcare",
        "Irregular Expenses", "Rent", "Savings", "Subscriptions", "Transport", "Utilities",
    ]
    assert categories == expected_categories
    profile_cols = ["user_id", "user_age", "user_monthly_income", "user_employment_status", "user_income_day"]
    profiles = data[profile_cols].drop_duplicates("user_id").set_index("user_id")
    rows = []
    weekly_feature_names = [
        "total_spending", "nonrec_spending", "recurring_spending", "transaction_count",
        "average_amount", *["category_" + c.lower().replace(" ", "_") for c in categories],
    ]
    for user_id, user_expense in expense.groupby("user_id", sort=True):
        weeks = pd.date_range(first_full, last_full, freq="W-MON")
        base = user_expense.groupby("period_start").agg(
            total_spending=("amount", "sum"), nonrec_spending=("nonrec_amount", "sum"),
            recurring_spending=("rec_amount", "sum"), transaction_count=("transaction_id", "size"),
            average_amount=("amount", "mean"),
        )
        cat = user_expense.pivot_table(index="period_start", columns="category", values="amount", aggfunc="sum", fill_value=0.0)
        weekly = pd.DataFrame(index=weeks).join(base).join(cat).fillna(0.0)
        for category in categories:
            if category not in weekly:
                weekly[category] = 0.0
        weekly = weekly.rename(columns={c: "category_" + c.lower().replace(" ", "_") for c in categories})
        weekly = weekly[weekly_feature_names]
        values = weekly.to_numpy(float)
        profile = profiles.loc[user_id]
        income = float(profile["user_monthly_income"])
        recurring_user = user_expense[user_expense["is_recurring"]]
        recurring_arrays = {}
        for category in RECURRING_CATEGORIES:
            category_rows = recurring_user[recurring_user["category"].eq(category)].sort_values("transaction_timestamp")
            recurring_arrays[category] = (
                pd.DatetimeIndex(category_rows["transaction_timestamp"]),
                category_rows["amount"].to_numpy(float),
            )
        for target_index in range(LOOKBACK, len(weekly)):
            target_start = weekly.index[target_index]
            target_end = target_start + pd.Timedelta(days=6)
            history = values[target_index - LOOKBACK:target_index]
            nonrec_history = history[:, weekly_feature_names.index("nonrec_spending")]
            total_history = history[:, weekly_feature_names.index("total_spending")]
            dynamic_recurring = 0.0
            recurring_context = []
            for category in RECURRING_CATEGORIES:
                category_times, category_amounts = recurring_arrays[category]
                cutoff = int(category_times.searchsorted(target_start, side="left"))
                features = next_due_features(
                    category_times[:cutoff], category_amounts[:cutoff], target_start, target_end, income
                )
                recurring_context.extend(features)
                dynamic_recurring += features[0] * features[1] * income * max(0.35, features[2])
            discretionary_baseline = float(np.median(nonrec_history[-4:]))
            dynamic_baseline = dynamic_recurring + discretionary_baseline
            month_angle = 2 * math.pi * (target_start.month - 1) / 12
            week_angle = 2 * math.pi * (target_start.isocalendar().week - 1) / 52
            dates = pd.date_range(target_start, target_end, freq="D")
            payday = int(profile["user_income_day"])
            prepayday_days = float(sum(((payday - int(day.day)) % 30) <= 3 for day in dates))
            status = str(profile["user_employment_status"])
            context = [
                income, float(profile["user_age"]), float(payday),
                float(status == "student"), float(status == "employed"), float(status == "self_employed"),
                prepayday_days, math.sin(month_angle), math.cos(month_angle),
                math.sin(week_angle), math.cos(week_angle),
                float(np.mean(total_history[-4:])), float(np.median(total_history[-4:])),
                float(np.std(total_history[-4:])), float(np.mean(nonrec_history[-4:])),
                float(np.median(nonrec_history[-4:])), float(dynamic_baseline),
                *recurring_context,
            ]
            rows.append({
                "source": source, "user": str(user_id), "date": target_start,
                "X": np.concatenate([history.reshape(-1), np.asarray(context, dtype=float)]),
                "y": float(values[target_index, weekly_feature_names.index("total_spending")]),
                "baseline": dynamic_baseline, "income": income,
            })
    return {
        "X": np.stack([row["X"] for row in rows]),
        "y": np.asarray([row["y"] for row in rows]),
        "baseline": np.asarray([row["baseline"] for row in rows]),
        "income": np.asarray([row["income"] for row in rows]),
        "date": pd.to_datetime([row["date"] for row in rows]).to_numpy(),
        "user": np.asarray([row["user"] for row in rows]),
        "source": np.asarray([row["source"] for row in rows]),
        "weekly_feature_names": weekly_feature_names,
    }


def metrics(actual: np.ndarray, prediction: np.ndarray) -> dict:
    prediction = np.maximum(0.0, np.asarray(prediction))
    error = prediction - actual
    return {
        "mae": float(np.abs(error).mean()),
        "rmse": float(mean_squared_error(actual, prediction) ** 0.5),
        "wape": float(100 * np.abs(error).sum() / np.abs(actual).sum()),
        "r2": float(r2_score(actual, prediction)),
        "bias": float(error.mean()),
    }


In [ ]:
forecast_started = time.perf_counter()
final_forecast = build_samples(final_seed, "locked_final_seed")
assert final_forecast["X"].shape[1] == int(feature_schema["forecast_feature_count"])
assert getattr(frozen_forecast_model, "n_features_in_", final_forecast["X"].shape[1]) == final_forecast["X"].shape[1]

# Prediction is frozen before any final metrics are calculated.
final_forecast_prediction = np.clip(
    frozen_forecast_model.predict(final_forecast["X"]) * final_forecast["income"],
    0.0,
    None,
)
final_forecast_metrics = metrics(final_forecast["y"], final_forecast_prediction)
forecast_checks = {
    "wape_at_or_below_target": bool(final_forecast_metrics["wape"] <= ACCEPTANCE["forecast_wape_max"]),
    "r2_at_or_above_target": bool(final_forecast_metrics["r2"] >= ACCEPTANCE["forecast_r2_min"]),
}
forecast_pass = all(forecast_checks.values())

forecast_predictions = pd.DataFrame({
    "user_id": final_forecast["user"],
    "week_start": pd.to_datetime(final_forecast["date"]),
    "actual_spending": final_forecast["y"],
    "predicted_spending": final_forecast_prediction,
    "monthly_income": final_forecast["income"],
})
forecast_predictions["error"] = forecast_predictions["predicted_spending"] - forecast_predictions["actual_spending"]
forecast_predictions["month"] = forecast_predictions["week_start"].dt.to_period("M").astype(str)
monthly_rows = []
for month, group in forecast_predictions.groupby("month", sort=True):
    monthly_rows.append({"month": month, **metrics(group["actual_spending"].to_numpy(), group["predicted_spending"].to_numpy())})
forecast_monthly_metrics = pd.DataFrame(monthly_rows)

display(pd.Series(final_forecast_metrics, name="final_value").to_frame())
display(forecast_monthly_metrics)
print({"forecast_pass": forecast_pass, "checks": forecast_checks, "seconds": time.perf_counter() - forecast_started})

plot_points = forecast_predictions.groupby("week_start", as_index=False)[["actual_spending", "predicted_spending"]].sum()
plt.figure(figsize=(15, 5))
plt.plot(plot_points["week_start"], plot_points["actual_spending"], label="Actual", linewidth=2)
plt.plot(plot_points["week_start"], plot_points["predicted_spending"], label="Frozen V2", linewidth=2)
plt.title("Final-seed weekly spending: actual vs frozen forecast")
plt.ylabel("KES")
plt.legend()
plt.tight_layout()
plt.show()


## 3. Frozen anomaly evaluation

The saved scaler and `if_4096` artifact are loaded unchanged. The 2024/original and Jan–Aug 2025
development features reconstruct only the isolation-score reference distribution used by V2;
no model is refit and no labels enter that calculation. The saved rule configuration, isolation
weight, and hybrid threshold then produce final predictions before labels are opened for metrics.


In [ ]:
def safe_ratio(numerator: float, denominator: float, neutral: float = 0.0) -> float:
    return float(numerator / denominator) if denominator and np.isfinite(denominator) else neutral


def build_past_only_features(raw: pd.DataFrame) -> pd.DataFrame:
    expense = raw[raw["transaction_type"].astype(str).str.lower().eq("expense")].copy()
    expense["transaction_timestamp"] = pd.to_datetime(expense["transaction_timestamp"])
    expense["amount"] = pd.to_numeric(expense["amount"], errors="coerce")
    expense["is_recurring"] = bools(expense["is_recurring"])
    expense["is_anomaly_target"] = bools(expense["is_anomaly"])
    expense = expense.sort_values(["user_id", "transaction_timestamp", "transaction_id"]).reset_index(drop=True)

    engineered_rows: list[dict] = []
    for user_id, group in expense.groupby("user_id", sort=False):
        q1h: deque[tuple[pd.Timestamp, float]] = deque()
        q24h: deque[tuple[pd.Timestamp, float]] = deque()
        q7d: deque[tuple[pd.Timestamp, float]] = deque()
        sum24h = 0.0
        sum7d = 0.0
        user_amounts: list[float] = []
        category_amounts: dict[str, list[float]] = defaultdict(list)
        recurring_amounts: dict[str, list[float]] = defaultdict(list)
        category_counts: dict[str, int] = defaultdict(int)
        last_exact: dict[tuple[str, str, float], pd.Timestamp] = {}
        last_merchant_category: dict[tuple[str, str], pd.Timestamp] = {}
        last_timestamp: pd.Timestamp | None = None
        day_key = None
        day_count = 0
        day_sum = 0.0

        for row in group.itertuples(index=False):
            timestamp = pd.Timestamp(row.transaction_timestamp)
            amount = float(row.amount)
            category = str(row.category)
            merchant = str(row.merchant)
            income = float(row.user_monthly_income)
            current_day = timestamp.date()
            if day_key != current_day:
                day_key = current_day
                day_count = 0
                day_sum = 0.0

            cut1h = timestamp - pd.Timedelta(hours=1)
            cut24h = timestamp - pd.Timedelta(days=1)
            cut7d = timestamp - pd.Timedelta(days=7)
            while q1h and q1h[0][0] < cut1h:
                q1h.popleft()
            while q24h and q24h[0][0] < cut24h:
                sum24h -= q24h.popleft()[1]
            while q7d and q7d[0][0] < cut7d:
                sum7d -= q7d.popleft()[1]

            user_mean = float(np.mean(user_amounts)) if user_amounts else np.nan
            user_median = float(np.median(user_amounts)) if user_amounts else np.nan
            cat_values = category_amounts[category]
            category_mean = float(np.mean(cat_values)) if cat_values else np.nan
            category_median = float(np.median(cat_values)) if cat_values else np.nan
            recurring_values = recurring_amounts[category]
            recurring_median = float(np.median(recurring_values)) if recurring_values else np.nan
            exact_key = (merchant, category, round(amount, 2))
            merchant_key = (merchant, category)
            exact_age = (
                (timestamp - last_exact[exact_key]).total_seconds() / 60.0
                if exact_key in last_exact else np.nan
            )
            merchant_age = (
                (timestamp - last_merchant_category[merchant_key]).total_seconds() / 60.0
                if merchant_key in last_merchant_category else np.nan
            )
            past_count = len(user_amounts)

            engineered_rows.append({
                "transaction_id": row.transaction_id,
                "log_amount": np.log1p(max(0.0, amount)),
                "amount_to_income": safe_ratio(amount, income),
                "amount_to_user_mean": safe_ratio(amount, user_mean, 1.0),
                "amount_to_user_median": safe_ratio(amount, user_median, 1.0),
                "amount_to_category_mean": safe_ratio(amount, category_mean, 1.0),
                "amount_to_category_median": safe_ratio(amount, category_median, 1.0),
                "recurring_amount_ratio": safe_ratio(amount, recurring_median, 1.0),
                "past_user_count": past_count,
                "category_proportion": safe_ratio(category_counts[category], past_count),
                "previous_1h_count": len(q1h),
                "previous_24h_count": len(q24h),
                "previous_7d_count": len(q7d),
                "previous_24h_sum_to_income": safe_ratio(sum24h, income),
                "previous_7d_sum_to_income": safe_ratio(sum7d, income),
                "same_day_count_before": day_count,
                "same_day_sum_to_income": safe_ratio(day_sum, income),
                "time_since_previous_hours": (
                    (timestamp - last_timestamp).total_seconds() / 3600.0 if last_timestamp is not None else np.nan
                ),
                "same_exact_age_minutes": exact_age,
                "same_merchant_category_age_minutes": merchant_age,
                "is_recurring_feature": int(bool(row.is_recurring)),
                "is_weekend": int(timestamp.dayofweek >= 5),
                "hour_sin": np.sin(2 * np.pi * timestamp.hour / 24),
                "hour_cos": np.cos(2 * np.pi * timestamp.hour / 24),
            })

            q1h.append((timestamp, amount))
            q24h.append((timestamp, amount))
            q7d.append((timestamp, amount))
            sum24h += amount
            sum7d += amount
            user_amounts.append(amount)
            category_amounts[category].append(amount)
            category_counts[category] += 1
            if bool(row.is_recurring):
                recurring_amounts[category].append(amount)
            last_exact[exact_key] = timestamp
            last_merchant_category[merchant_key] = timestamp
            last_timestamp = timestamp
            day_count += 1
            day_sum += amount

    engineered = pd.DataFrame(engineered_rows)
    return expense.merge(engineered, on="transaction_id", how="left", validate="one_to_one")


def split_chronologically(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    unique_times = np.sort(frame["transaction_timestamp"].unique())
    train_count = max(1, int(np.floor(len(unique_times) * 0.70)))
    train_end = unique_times[train_count - 1]
    train = frame[frame["transaction_timestamp"] <= train_end].copy()
    held = frame[frame["transaction_timestamp"] > train_end].copy()
    held_times = np.sort(held["transaction_timestamp"].unique())
    val_count = max(1, int(np.floor(len(held_times) * 0.20)))
    val_end = held_times[val_count - 1]
    validation = held[held["transaction_timestamp"] <= val_end].copy()
    evaluation = held[held["transaction_timestamp"] > val_end].copy()
    return train, validation, evaluation


def anomaly_metric_record(actual: np.ndarray, predicted: np.ndarray) -> dict:
    actual = np.asarray(actual, dtype=bool)
    predicted = np.asarray(predicted, dtype=bool)
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[False, True]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, average="binary", zero_division=0
    )
    return {
        "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "false_positive_rate": float(fp / (fp + tn)), "tp": int(tp), "fp": int(fp),
        "tn": int(tn), "fn": int(fn), "alerts": int(predicted.sum()),
    }


def best_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, dict]:
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    if not len(thresholds):
        threshold = float(np.max(scores) + 1)
    else:
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, anomaly_metric_record(labels, scores >= threshold)


def rules(frame: pd.DataFrame, config: dict) -> tuple[np.ndarray, np.ndarray]:
    duplicate = frame["same_exact_age_minutes"].between(0, config["duplicate_minutes"], inclusive="both")
    burst = frame["previous_1h_count"].ge(config["burst_count"])
    recurring = (
        frame["is_recurring_feature"].eq(1)
        & frame["category"].eq("Rent")
        & frame["recurring_amount_ratio"].ge(config["recurring_ratio"])
        & frame["past_user_count"].ge(20)
    )
    weekend = (
        frame["is_weekend"].eq(1)
        & frame["category"].eq("Entertainment")
        & frame["amount_to_income"].ge(config["weekend_ratio"])
    )
    random_shock = (
        frame["category"].eq("Irregular Expenses")
        & frame["amount_to_income"].ge(config["shock_ratio"])
    )
    extreme = frame["amount_to_income"].ge(config["extreme_ratio"])
    rare = (
        frame["past_user_count"].ge(30)
        & frame["category_proportion"].le(config["rare_proportion"])
        & frame["amount_to_income"].ge(config["rare_amount_ratio"])
    )
    flags = np.column_stack([duplicate, burst, recurring, weekend, random_shock, extreme, rare]).astype(int)
    weights = np.array([3.0, 2.0, 3.0, 2.0, 3.0, 3.0, 2.0])
    return flags.dot(weights), flags


def percentile_against(reference: np.ndarray, values: np.ndarray) -> np.ndarray:
    ordered = np.sort(np.asarray(reference, dtype=float))
    return np.searchsorted(ordered, values, side="right") / max(1, len(ordered))


def add_cluster_features(frame: pd.DataFrame) -> pd.DataFrame:
    outputs = []
    for _, group in frame.groupby("user_id", sort=False):
        queue = deque()
        past_count = 0
        past_night_count = 0
        for row in group.sort_values(["transaction_timestamp", "transaction_id"]).itertuples(index=False):
            timestamp = pd.Timestamp(row.transaction_timestamp)
            cutoff = timestamp - pd.Timedelta(hours=1)
            while queue and queue[0][0] < cutoff:
                queue.popleft()
            same = [item for item in queue if item[2] == str(row.merchant) and item[3] == str(row.category)]
            income = max(1.0, float(row.user_monthly_income))
            outputs.append({
                "transaction_id": row.transaction_id,
                "distinct_merchants_1h": float(len({item[2] for item in queue})),
                "same_merchant_category_count_1h": float(len(same)),
                "total_amount_with_current_1h_to_income": float((sum(item[1] for item in queue) + float(row.amount)) / income),
                "same_merchant_category_sum_with_current_1h_to_income": float((sum(item[1] for item in same) + float(row.amount)) / income),
                "historical_night_proportion": float(past_night_count / past_count) if past_count else 0.0,
                "is_night": int(timestamp.hour <= 5),
                "is_known_recurring_category": int(str(row.category) in KNOWN_RECURRING),
            })
            queue.append((timestamp, float(row.amount), str(row.merchant), str(row.category)))
            past_count += 1
            past_night_count += int(timestamp.hour <= 5)
    return frame.merge(pd.DataFrame(outputs), on="transaction_id", how="left", validate="one_to_one")


def v2_rule_scores(frame: pd.DataFrame, config: dict) -> tuple[np.ndarray, np.ndarray]:
    nonrecurring = frame["is_recurring_feature"].eq(0)
    duplicate = frame["same_exact_age_minutes"].between(0, config["duplicate_minutes"], inclusive="both")
    burst = (
        frame["previous_1h_count"].ge(config["burst_count"])
        & frame["total_amount_with_current_1h_to_income"].ge(config["burst_ratio"])
    )
    recurring_increase = (
        frame["is_recurring_feature"].eq(1)
        & frame["is_known_recurring_category"].eq(1)
        & frame["recurring_amount_ratio"].ge(config["recurring_ratio"])
        & frame["past_user_count"].ge(20)
    )
    weekend = (
        frame["is_weekend"].eq(1)
        & frame["category"].eq("Entertainment")
        & frame["amount_to_income"].ge(config["weekend_ratio"])
    )
    random_shock = (
        frame["category"].eq("Irregular Expenses")
        & frame["amount_to_income"].ge(config["shock_ratio"])
    )
    extreme = nonrecurring & frame["amount_to_income"].ge(config["extreme_ratio"])
    rare = (
        nonrecurring
        & frame["is_known_recurring_category"].eq(0)
        & frame["past_user_count"].ge(30)
        & frame["category_proportion"].le(config["rare_proportion"])
        & frame["amount_to_income"].ge(config["rare_amount_ratio"])
    )
    merchant_hopping = (
        frame["previous_1h_count"].ge(2)
        & frame["distinct_merchants_1h"].ge(config["distinct_merchants"])
        & frame["total_amount_with_current_1h_to_income"].ge(config["merchant_hopping_ratio"])
    )
    split_pattern = (
        frame["same_merchant_category_count_1h"].ge(2)
        & frame["same_merchant_category_sum_with_current_1h_to_income"].ge(config["split_ratio"])
    )
    nighttime = (
        frame["is_night"].eq(1)
        & frame["historical_night_proportion"].le(config["night_history_max"])
        & frame["amount_to_income"].ge(config["night_ratio"])
    )
    flags = np.column_stack([
        duplicate, burst, recurring_increase, weekend, random_shock,
        extreme, rare, merchant_hopping, split_pattern, nighttime,
    ]).astype(int)
    weights = np.array([3.0, 2.0, 3.0, 2.0, 3.0, 3.0, 2.0, 2.5, 3.0, 3.0])
    return flags.dot(weights), flags


In [ ]:
anomaly_started = time.perf_counter()
anomaly_features = list(feature_schema["anomaly_features"])
anomaly_fill_values = {key: float(value) for key, value in feature_schema["anomaly_fill_values"].items()}
frozen_rule_config = feature_schema["anomaly_rule_config"]
frozen_if_weight = float(feature_schema["anomaly_if_weight"])
frozen_hybrid_threshold = float(feature_schema["anomaly_hybrid_threshold"])

# Reconstruct the V2 reference score distribution without using any labels.
original_anomaly_frame = add_cluster_features(build_past_only_features(original))
original_anomaly_train, _, _ = split_chronologically(original_anomaly_frame)
development_anomaly_frame = add_cluster_features(build_past_only_features(development_stress))
development_reference = development_anomaly_frame[
    development_anomaly_frame["transaction_timestamp"] < pd.Timestamp("2025-09-01")
].copy()
reference_frame = pd.concat([original_anomaly_train, development_reference], ignore_index=True)
reference_matrix = reference_frame.reindex(columns=anomaly_features).fillna(anomaly_fill_values).fillna(0.0).to_numpy(float)
reference_scaled = frozen_anomaly_scaler.transform(reference_matrix)
reference_if_scores = -frozen_isolation_forest.score_samples(reference_scaled)

final_anomaly_frame = add_cluster_features(build_past_only_features(final_seed))
final_matrix = final_anomaly_frame.reindex(columns=anomaly_features).fillna(anomaly_fill_values).fillna(0.0).to_numpy(float)
final_scaled = frozen_anomaly_scaler.transform(final_matrix)
final_if_scores = -frozen_isolation_forest.score_samples(final_scaled)
final_if_percentiles = percentile_against(reference_if_scores, final_if_scores)
final_rule_scores, final_rule_flags = v2_rule_scores(final_anomaly_frame, frozen_rule_config)
final_hybrid_scores = final_rule_scores + frozen_if_weight * final_if_percentiles

# Predictions are now locked; labels are opened only below for evaluation.
final_anomaly_prediction = final_hybrid_scores >= frozen_hybrid_threshold
final_anomaly_labels = final_anomaly_frame["is_anomaly_target"].astype(bool).to_numpy()
final_anomaly_metrics = anomaly_metric_record(final_anomaly_labels, final_anomaly_prediction)
anomaly_checks = {
    "precision_at_or_above_target": bool(final_anomaly_metrics["precision"] >= ACCEPTANCE["anomaly_precision_min"]),
    "recall_at_or_above_target": bool(final_anomaly_metrics["recall"] >= ACCEPTANCE["anomaly_recall_min"]),
}
anomaly_pass = all(anomaly_checks.values())

anomaly_predictions = final_anomaly_frame[[
    "transaction_id", "user_id", "transaction_timestamp", "amount", "category",
    "merchant", "anomaly_type", "is_anomaly_target",
]].copy()
anomaly_predictions["rule_score"] = final_rule_scores
anomaly_predictions["isolation_percentile"] = final_if_percentiles
anomaly_predictions["hybrid_score"] = final_hybrid_scores
anomaly_predictions["predicted_anomaly"] = final_anomaly_prediction

scenario_metrics = (
    anomaly_predictions[anomaly_predictions["is_anomaly_target"]]
    .groupby("anomaly_type")["predicted_anomaly"].agg(["size", "sum"])
    .reset_index().rename(columns={"size": "labelled", "sum": "detected"})
)
scenario_metrics["recall"] = scenario_metrics["detected"] / scenario_metrics["labelled"]
false_positive_categories = (
    anomaly_predictions[
        anomaly_predictions["predicted_anomaly"] & ~anomaly_predictions["is_anomaly_target"]
    ]["category"].value_counts().rename("false_positives").reset_index()
)

display(pd.Series(final_anomaly_metrics, name="final_value").to_frame())
display(scenario_metrics)
display(false_positive_categories)
print({"anomaly_pass": anomaly_pass, "checks": anomaly_checks, "seconds": time.perf_counter() - anomaly_started})

matrix = confusion_matrix(final_anomaly_labels, final_anomaly_prediction, labels=[False, True])
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Frozen V2 anomaly model on final seed")
plt.tight_layout()
plt.show()


## 4. Final decision and evidence export

All four original production targets must pass on this locked seed. A pass makes the models
eligible for app-integration review; it does not silently modify the app. A failure keeps the app
unchanged and identifies the area requiring another explicitly versioned development cycle.


In [ ]:
overall_pass = bool(forecast_pass and anomaly_pass)
if overall_pass:
    recommendation = "eligible_for_app_integration_review"
    next_action = "Review app data contracts, package the frozen artifacts, and add monitored shadow-mode integration."
else:
    failed_areas = []
    if not forecast_pass:
        failed_areas.append("forecast")
    if not anomaly_pass:
        failed_areas.append("anomaly")
    recommendation = "continue_model_development_before_app_integration"
    next_action = f"Keep the app unchanged and begin a new versioned development cycle for: {', '.join(failed_areas)}."

decision_table = pd.DataFrame([
    {"area": "dataset", "check": "new users vs original", "status": dataset_checks["new_users_vs_original"]},
    {"area": "dataset", "check": "new users vs development", "status": dataset_checks["new_users_vs_development"]},
    {"area": "dataset", "check": "12-month coverage", "status": dataset_checks["full_year_coverage"]},
    {"area": "forecast", "check": f"WAPE <= {ACCEPTANCE['forecast_wape_max']}", "status": forecast_checks["wape_at_or_below_target"]},
    {"area": "forecast", "check": f"R2 >= {ACCEPTANCE['forecast_r2_min']}", "status": forecast_checks["r2_at_or_above_target"]},
    {"area": "anomaly", "check": f"precision >= {ACCEPTANCE['anomaly_precision_min']}", "status": anomaly_checks["precision_at_or_above_target"]},
    {"area": "anomaly", "check": f"recall >= {ACCEPTANCE['anomaly_recall_min']}", "status": anomaly_checks["recall_at_or_above_target"]},
    {"area": "governance", "check": "frozen artifacts; no fit or tuning", "status": True},
    {"area": "governance", "check": "final labels unused before prediction", "status": True},
    {"area": "governance", "check": "app unchanged", "status": True},
])
display(decision_table)
print({"recommendation": recommendation, "next_action": next_action})

forecast_predictions.to_csv(OUTPUT_DIR / "final_forecast_predictions.csv", index=False)
forecast_monthly_metrics.to_csv(OUTPUT_DIR / "final_forecast_monthly_metrics.csv", index=False)
anomaly_predictions.to_csv(OUTPUT_DIR / "final_anomaly_predictions.csv", index=False)
scenario_metrics.to_csv(OUTPUT_DIR / "final_anomaly_scenario_metrics.csv", index=False)
false_positive_categories.to_csv(OUTPUT_DIR / "final_anomaly_false_positive_categories.csv", index=False)
decision_table.to_csv(OUTPUT_DIR / "final_acceptance_checks.csv", index=False)

result_payload = {
    "notebook": "Spending_Model_Final_Seed_Test",
    "protocol": {
        "v2_evidence_sha256": v2_evidence_sha256,
        "final_seed_derivation": "first_32_bits_sha256(original_sha256|v2_evidence_sha256|SPENDLY_FINAL_V2_ONCE)",
        "final_seed": FINAL_SEED,
        "frozen_forecast_model": v2_results["frozen_forecast_model"],
        "frozen_anomaly_if": v2_results["frozen_anomaly_if"],
        "frozen_anomaly_rule_config": frozen_rule_config,
        "frozen_anomaly_if_weight": frozen_if_weight,
        "frozen_anomaly_hybrid_threshold": frozen_hybrid_threshold,
        "artifact_hashes": artifact_hashes,
    },
    "final_dataset": final_seed_summary,
    "acceptance_targets": ACCEPTANCE,
    "forecast_final_metrics": final_forecast_metrics,
    "forecast_checks": forecast_checks,
    "forecast_pass": forecast_pass,
    "forecast_monthly_metrics": forecast_monthly_metrics.round(6).to_dict("records"),
    "anomaly_final_metrics": final_anomaly_metrics,
    "anomaly_checks": anomaly_checks,
    "anomaly_pass": anomaly_pass,
    "scenario_metrics": scenario_metrics.round(6).to_dict("records"),
    "overall_pass": overall_pass,
    "recommendation": recommendation,
    "next_action": next_action,
    "checks": {
        **dataset_checks,
        "input_hashes_validated": True,
        "frozen_artifacts_loaded": True,
        "no_model_fit_or_threshold_search": True,
        "final_seed_labels_unused_for_tuning": True,
        "final_seed_locked_before_scoring": True,
        "app_changed": False,
    },
}
results_path = OUTPUT_DIR / "spendly_final_seed_results.json"
results_path.write_text(json.dumps(result_payload, indent=2, default=str), encoding="utf-8")

archive_base = Path("spendly_final_seed_output")
zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
print("SPENDLY_FINAL_SEED_RESULT_START")
print(json.dumps(result_payload, indent=2, default=str))
print("SPENDLY_FINAL_SEED_RESULT_END")
print(f"Saved: {zip_path.resolve()}")
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass


## Interpretation

- `overall_pass: true` means the frozen V2 configuration cleared every predeclared target on the
  untouched final seed and can move to a separate app-integration review.
- `overall_pass: false` means no app integration yet. Do not retune against this final dataset;
  preserve it as evidence and open a newly versioned development cycle.
